In [1]:
import os
import sys
import argparse

import numpy as np
import matplotlib.pyplot as plt

import torch

import dolfinx
import dolfinx.fem.petsc
import ufl
from mpi4py import MPI
import basix.ufl

repo_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(repo_path)

from utils import load_yaml, load_pkl, load_npy, format_elapsed_time, timing, plot_complex_valued_function, plot_real_valued_function, evaluate_expression

from scifem import create_real_functionspace

import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
from petsc4py import PETSc
import scifem


from utils import project, norm_L2, convert_petsc_mat_to_torch_sparse_coo_tensor, convert_weight_to_tensor
from typing import Optional
import pickle
import time
from tqdm import tqdm
import matplotlib.ticker as ticker

from data_generation.differential_equations import PoissonSetup2LeastSquares

----------------------------------------
2025-11-25 22:34:58 - Start Program
----------------------------------------


In [2]:
mesh_args = {
    "lower_left_x": 0.0,      # x-coordinate of the lower-left corner
    "lower_left_y": 0.0,      # y-coordinate of the lower-left corner
    "upper_right_x": 1.0,     # x-coordinate of the upper-right corner
    "upper_right_y": 1.0,     # y-coordinate of the upper-right corner
    "num_x": 256,             # Number of cells along x-axis
    "num_y": 256,             # Number of cells along y-axis
    "mesh_cell_type": "triangle"  # "triangle" or "quadrilateral"
}

In [22]:
function_space_args = {
    "m": {
        "family": "CG",
        "degree": 1
    },
    "p": {
        "family": "CG",
        "degree": 1
    },
    "u": {
        "family": "CG",
        "degree": 2
    },
    "sigma": {
        "family": "RT",
        "degree": 2
    },
    "w": {
        "family": "CG",
        "degree": 2
    },
    "q": {
        "family": "CG",
        "degree": 2
    }
}

In [4]:
function_space_finer_args = {
    "m": {
        "family": "CG",
        "degree": 1
    },
    "p": {
        "family": "CG",
        "degree": 1
    },
    "u": {
        "family": "CG",
        "degree": 4
    },
    "sigma": {
        "family": "RT",
        "degree": 4
    },
    "w": {
        "family": "CG",
        "degree": 4
    },
    "q": {
        "family": "CG",
        "degree": 4
    }
}

In [5]:
print(f'mesh args: {mesh_args}')
print(f'function space args: {function_space_args}')
print(f'function space finer args: {function_space_finer_args}')

mesh args: {'lower_left_x': 0.0, 'lower_left_y': 0.0, 'upper_right_x': 1.0, 'upper_right_y': 1.0, 'num_x': 256, 'num_y': 256, 'mesh_cell_type': 'triangle'}
function space args: {'m': {'family': 'CG', 'degree': 1}, 'p': {'family': 'CG', 'degree': 1}, 'u': {'family': 'CG', 'degree': 1}, 'sigma': {'family': 'RT', 'degree': 1}, 'w': {'family': 'CG', 'degree': 1}, 'q': {'family': 'CG', 'degree': 1}}
function space finer args: {'m': {'family': 'CG', 'degree': 1}, 'p': {'family': 'CG', 'degree': 1}, 'u': {'family': 'CG', 'degree': 4}, 'sigma': {'family': 'RT', 'degree': 4}, 'w': {'family': 'CG', 'degree': 4}, 'q': {'family': 'CG', 'degree': 4}}


In [23]:
poisson_least_squares = PoissonSetup2LeastSquares(mesh_args, function_space_args)
poisson_least_squares_finer = PoissonSetup2LeastSquares(mesh_args, function_space_finer_args)

In [24]:
mesh = poisson_least_squares.mesh
Vh = poisson_least_squares.Vh

In [25]:
# Make sure two function spaces are defined on the same mesh
u_element = basix.ufl.element(family=function_space_finer_args["u"]["family"],
                                cell=mesh_args["mesh_cell_type"],
                                degree=function_space_finer_args["u"]["degree"])
sigma_element = basix.ufl.element(family=function_space_finer_args["sigma"]["family"],
                                    cell=mesh_args["mesh_cell_type"],
                                    degree=function_space_finer_args["sigma"]["degree"])
sigma_u_element = basix.ufl.mixed_element([sigma_element, u_element])
finer_Vh = {
    'sigma_u': dolfinx.fem.functionspace(mesh, sigma_u_element)
}

In [9]:
num_samples = 100

In [10]:
train_dataset_path = repo_path + "/results/poisson_setup2/train_dataset"
test_dataset_path = repo_path + "/results/poisson_setup2/test_dataset"

In [11]:
if mesh_args['num_x'] == 128:
    p_dof = np.load(test_dataset_path + "/p_dof.npy")[:num_samples]
elif mesh_args['num_x'] == 256:
    p_dof = np.load(test_dataset_path + "/p_dof_256x256.npy")[:num_samples]

In [26]:
# Solve auxiliary problems
poisson_least_squares.q = poisson_least_squares.solve_q()
poisson_least_squares.w = poisson_least_squares.solve_w()

In [27]:
dtype = 'float64'
sigma_u_dim = len(dolfinx.fem.Function(Vh['sigma_u']).x.array)
sigma_u_dof = np.zeros((num_samples, sigma_u_dim), dtype=dtype)

time_for_solving_PDEs = []
for i in tqdm(range(num_samples)):
    start_time = time.time()
    p = dolfinx.fem.Function(poisson_least_squares.Vh['p'], dtype=dtype)
    p.x.array[:] = p_dof[i,:]
    sigma_u = poisson_least_squares.solve_sigma_u(p=p)
    sigma_u_dof[i,:] = sigma_u.x.array
    end_time = time.time()
    time_for_solving_PDEs.append(end_time - start_time)

100%|██████████| 100/100 [13:55<00:00,  8.36s/it]


In [28]:
print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs):.2f} seconds')
print(f'average time per sample: {np.mean(time_for_solving_PDEs):.2f} seconds')

Solved for 100 samples in 835.86 seconds
average time per sample: 8.36 seconds


In [15]:
# Solve auxiliary problems for finer space
poisson_least_squares_finer.q = poisson_least_squares_finer.solve_q()
poisson_least_squares_finer.w = poisson_least_squares_finer.solve_w()

In [16]:
dtype = 'float64'
sigma_u_finer_dim = len(dolfinx.fem.Function(finer_Vh['sigma_u']).x.array)
sigma_u_finer_dof = np.zeros((num_samples, sigma_u_finer_dim), dtype=dtype)

time_for_solving_PDEs_finer = []
for i in tqdm(range(num_samples)):
    start_time = time.time()
    p_finer = dolfinx.fem.Function(poisson_least_squares_finer.Vh['p'], dtype=dtype)
    p_finer.x.array[:] = p_dof[i,:]
    sigma_u_finer = poisson_least_squares_finer.solve_sigma_u(p=p_finer)
    sigma_u_finer_dof[i,:] = sigma_u_finer.x.array
    end_time = time.time()
    time_for_solving_PDEs_finer.append(end_time - start_time)

100%|██████████| 100/100 [1:09:00<00:00, 41.41s/it]


In [17]:
print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs_finer):.2f} seconds')
print(f'average time per sample: {np.mean(time_for_solving_PDEs_finer):.2f} seconds')

Solved for 100 samples in 4140.59 seconds
average time per sample: 41.41 seconds


In [18]:
compute_squared_hdiv_h1_norm = poisson_least_squares.compute_squared_hdiv_h1_norm

In [19]:
torch_dtype = {
    'float32': torch.float32,
    'float64': torch.float64
}

In [29]:
squared_error_list = []
reference_loss_list = []
time_for_assembling_weight = []
for test_index in tqdm(range(num_samples)):
    sigma_u_fc = dolfinx.fem.Function(Vh['sigma_u'])
    sigma_u_fc.x.array[:] = sigma_u_dof[test_index]
    sigma_fc = sigma_u_fc.sub(0).collapse()
    u_fc = sigma_u_fc.sub(1).collapse()

    sigma_u_finer_fc = dolfinx.fem.Function(finer_Vh['sigma_u'])
    sigma_u_finer_fc.x.array[:] = sigma_u_finer_dof[test_index]
    sigma_finer_fc = sigma_u_finer_fc.sub(0).collapse()
    u_finer_fc = sigma_u_finer_fc.sub(1).collapse()

    squared_error = compute_squared_hdiv_h1_norm(sigma_fc - sigma_finer_fc, u_fc - u_finer_fc)

    p_fc = dolfinx.fem.Function(Vh['p'])  
    p_fc.x.array[:] = p_dof[test_index]

    start_time = time.time()
    weight = poisson_least_squares.compute_weight(p_fc)
    end_time = time.time()
    time_for_assembling_weight.append(end_time - start_time)

    weight_tensor = convert_weight_to_tensor(weight, dtype=torch_dtype['float64'])

    y = sigma_u_dof[test_index]
    y = torch.tensor(y, dtype=torch_dtype['float64'])
    reference_loss = torch.dot(y, weight_tensor['A00'] @ y) + 2*torch.dot(y, weight_tensor['A01'])  + weight_tensor['A11']

    squared_error_list.append(squared_error)
    reference_loss_list.append(reference_loss.item())

100%|██████████| 100/100 [11:48<00:00,  7.08s/it]


In [31]:
print(f'# DoFs of sigma_u: {sigma_u_dof.shape[1]}')
print(f'# DoFs of finer sigma_u: {sigma_u_finer_dof.shape[1]}')
print(f'average time for assembling weight: {np.round(np.mean(time_for_assembling_weight), decimals=2)} seconds')
print(f'mean squared error between coarse and finer solutions: {np.mean(squared_error_list):.2e}')
print(f'mean reference loss: {np.mean(reference_loss_list):.2e}')

# DoFs of sigma_u: 919553
# DoFs of finer sigma_u: 3411969
average time for assembling weight: 1.88 seconds
mean squared error between coarse and finer solutions: 2.53e-04
mean reference loss: 2.53e-04
